<a href="https://colab.research.google.com/github/lestojas/segmentation/blob/matanglawin-dataset/colab/train_crack_direction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crack Detection, Segmentation & Direction Classification

This notebook trains and evaluates **three YOLO-seg models and three YOLO-cls
models** -- `yolo11n`, `yolo11m` (two YOLO11 sizes), and `yolov8n` (a lighter,
previous-generation model, included because YOLO11n is already the smallest
official YOLO11 size -- going lighter still means a different generation, not just
a smaller YOLO11 checkpoint) -- on the `lestojas/segmentation` crack dataset. All
three are trained and evaluated **identically** -- same data, same
hyperparameters, same epoch budget -- and none of them is treated as a default,
"main," or baseline model. Comparing across genuinely different model
configurations (not just sizes of one architecture) is exactly what the paired
significance tests in Table 7 are designed to test for:

> **⚠️ Direction ground truth was corrected.** The original direction labeler pooled every segmentation-polygon vertex from every crack fragment in an image into one point cloud before computing its principal axis, which conflated a fragment's own orientation with where fragments happened to sit in the frame -- confirmed to mislabel some images (e.g. three separate vertical hairline cracks scattered across a frame read as "Horizontal"). It now computes each fragment's own PCA angle and combines fragments with a weighted circular mean, falling back to "Mixed" when fragments disagree by 60° or more (see `DIRECTION_LABELS.md`). **99 of 870 crack images (~11%) changed label.** Segmentation polygons/masks were not touched -- only `-cls` (direction) training is affected. If you have `crack_direction_cls_*` checkpoints saved from before this fix, they were trained on the old labels and should be retrained, not recovered (see Section 0 below).
>
1. **-seg** (`yolo11n-seg` / `yolo11m-seg` / `yolov8n-seg`) — instance
   segmentation. Answers *"is there a crack, and what shape is it?"* (crack vs.
   no-crack detection + pixel-accurate mask segmentation, jointly -- a `-seg`
   checkpoint always does both).
2. **-cls** (`yolo11n-cls` / `yolo11m-cls` / `yolov8n-cls`) — image
   classification. Answers *"which direction does the crack run?"* (Horizontal /
   Vertical / Diagonal / Mixed), trained on the direction labels already computed in
   the repo (`*/_direction_labels.csv`, derived via PCA on the ground-truth
   segmentation polygons — see `DIRECTION_LABELS.md`).

Every training run is followed by a **safety checkpoint**: it bundles whatever's
been trained so far into an organized, uniquely-named zip and triggers a real
browser download to your computer's normal Downloads folder, and (optionally, if
you enter a GitHub token in Section 0) also commits and pushes the same weights to
`trained_weights/` in this repo -- so a runtime disconnect never costs you an
already-trained model's weights, only its retraining time. Training cells also skip
straight to evaluation if a matching `runs/<run_name>/weights/best.pt` already
exists (see Section 0 for how to re-upload previously-trained weights to make use
of this).

A final evaluation section reports all three requested accuracies, for every model:
- **Crack vs. no-crack detection accuracy**
- **Crack shape segmentation accuracy** (mask IoU / mAP)
- **Crack direction classification accuracy**

Section 10 then packages the key results into **eight research-paper-ready tables**
(Table 0: image counts per split, for a Methods section; Table 1: the test split's
composition in detail, for a Results section; Tables 2-6: descriptive performance --
detection confusion matrix, detection image-level metrics, detection box-level
metrics, segmentation, direction, one column per model; Table 7: symmetric,
two-sided pairwise significance tests between every pair of models -- 9 rows, no
model singled out as a reference) — exported as CSV and LaTeX and zipped for
download.

> **ℹ️ About the negative images:** every split includes genuine crack-free
> negatives (see `DIRECTION_LABELS.md`) — real crops from a crack-containing photo's
> crack-free region, not synthetic images. `train`/`valid` have ~15% negatives; the
> held-out **test** split is rebalanced to near-parity (89 crack / 89 crack-free) so
> the detection confusion matrix (TP/FP/FN/TN) and specificity are meaningful
> rather than computed from a handful of negatives.

**Before running:** `Runtime → Change runtime type → GPU` (T4 or better).

## 0. Setup

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected — go to Runtime > Change runtime type > GPU before continuing.')

In [ ]:
!pip install -q ultralytics scikit-learn seaborn

In [ ]:
# --- Configuration ---------------------------------------------------------
REPO_URL = 'https://github.com/lestojas/segmentation.git'
BRANCH   = 'matanglawin-dataset'  # switch to 'main' once this branch is merged

# All models here are trained and evaluated identically -- none is a "main" model or
# a "baseline." Two YOLO11 sizes plus a YOLOv8 nano model as a genuinely lighter-weight
# third option (YOLO11n is already the smallest official YOLO11 size, so going lighter
# still means a different generation, not just a smaller YOLO11 checkpoint). Add or
# remove entries here to change which models are compared -- everything downstream
# (training, evaluation, checkpoints, tables) adapts automatically to however many are listed.
SEG_MODELS = ['yolo11n-seg.pt', 'yolo11m-seg.pt', 'yolov8n-seg.pt']
CLS_MODELS = ['yolo11n-cls.pt', 'yolo11m-cls.pt', 'yolov8n-cls.pt']

IMG_SIZE    = 640
SEG_EPOCHS  = 50
CLS_EPOCHS  = 50
TRAIN_PATIENCE = 10  # epochs of no val improvement before early stopping
BATCH       = -1   # -1 = Ultralytics auto-picks the largest batch that fits in GPU memory
CONF_THRES  = 0.25 # confidence threshold used for the 'is a crack detected in this image' metric

FORCE_RETRAIN = False  # if runs/<run_name>/weights/best.pt already exists for a given model (e.g. you
                        # uploaded a previously-trained checkpoint, or re-ran this notebook in the same
                        # session), training for that model is skipped and the existing weights are reused
                        # as-is. Set True to always retrain every model from scratch regardless.

SAVE_TO_DRIVE = False  # set True to copy trained weights to your Google Drive at the end

### Recovering previously-trained weights (optional -- skips retraining)

Every training cell in Sections 3-4 checks whether `runs/<run_name>/weights/best.pt`
already exists before training, and skips straight past it if so (set
`FORCE_RETRAIN = True` above to always retrain regardless). If you already trained
some of these models in an earlier session and still have the `.pt` files saved
somewhere, you can skip retraining just those models:

1. Run the repo-clone cell below first, so the working directory exists to upload into.
2. For each model you want to skip, create its `weights/` folder and drop `best.pt`
   into it, e.g. for `yolo11m-seg`:
   ```python
   import os; os.makedirs('runs/crack_seg_yolo11m/weights', exist_ok=True)
   from google.colab import files
   uploaded = files.upload()  # pick your saved best.pt in the dialog
   os.rename(list(uploaded.keys())[0], 'runs/crack_seg_yolo11m/weights/best.pt')
   ```
   The run names to match are `crack_seg_yolo11n` / `crack_seg_yolo11m` /
   `crack_seg_yolov8n` for the seg models, and `crack_direction_cls_yolo11n` /
   `crack_direction_cls_yolo11m` / `crack_direction_cls_yolov8n` for the cls models.
3. If you used `SAVE_TO_DRIVE = True` in an earlier run instead, mount Drive
   (`from google.colab import drive; drive.mount('/content/drive')`) and copy from
   `MyDrive/crack_models/weights/` into the matching
   `runs/<run_name>/weights/best.pt` path.

Either way, Sections 3-4 work correctly either path: found weights are reused as-is,
anything missing is trained normally.

> **Direction ground truth changed (see the warning at the top of this notebook).** Only recover `crack_seg_*` checkpoints this way -- those depend solely on the segmentation polygons, which were not touched by the fix. Do **not** recover `crack_direction_cls_*` checkpoints trained before the fix; leave those out so Section 4 retrains them against the corrected labels.


In [ ]:
import os, shutil
DATA_DIR = '/content/segmentation'
if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
!git clone --branch {BRANCH} --single-branch {REPO_URL} {DATA_DIR}
os.chdir(DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
import json, csv, math, random, zipfile, itertools, re, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw
from sklearn.metrics import confusion_matrix, classification_report
from scipy import stats
from IPython.display import display

from ultralytics import YOLO

SPLITS = ['train', 'valid', 'test']
random.seed(0)

### (Optional) Also back up trained weights to this GitHub repo

Besides the local browser-download safety checkpoints (Sections 3-4 below), this
notebook can also commit each trained model's `best.pt` straight to this repo under
`trained_weights/`, pushed to `BRANCH`, as a second off-runtime backup -- so even if
you close this tab entirely, the weights are recoverable from GitHub, not just your
Downloads folder.

This needs a GitHub **Personal Access Token** with `repo` write scope
(GitHub -> Settings -> Developer settings -> Personal access tokens -> generate one
-> scope: `repo`). It's read with `getpass` below, so it's never echoed to the cell
output and never written into this notebook file -- it only lives in this runtime's
memory for this session. **Leave the prompt empty and press Enter to skip this
feature entirely** -- everything else works identically either way; you'd just rely
on the browser downloads alone for backup.

Note: individual weight files here run from a few MB up to ~40-50MB (`yolo11m-*`) --
under GitHub's 100MB hard limit, but large enough that GitHub may show a "large
files" warning on push. That's expected and harmless for a one-time backup like this;
if you plan to do this repeatedly across many training runs, consider Git LFS instead.

In [ ]:
GITHUB_TOKEN = getpass('GitHub Personal Access Token (repo write scope) -- leave blank to skip GitHub backups: ')
PUSH_WEIGHTS_TO_GITHUB = bool(GITHUB_TOKEN.strip())

if PUSH_WEIGHTS_TO_GITHUB:
    print('GitHub backup enabled -- trained weights will be committed to trained_weights/ and pushed to', BRANCH)
else:
    print('No token entered -- GitHub backup disabled. Local browser-download checkpoints (Sections 3-4, 11) still apply.')

## 1. Convert COCO segmentation annotations → YOLO-seg format

Ultralytics expects one `.txt` label file per image, one line per instance:
`class_id x1 y1 x2 y2 ... xn yn` with all coordinates normalized to `[0, 1]`. This
dataset has a single object class (`Cracks`), so every instance maps to class `0`.

In [ ]:
YOLO_SEG_DIR = Path('/content/yolo_seg_dataset')
if YOLO_SEG_DIR.exists():
    shutil.rmtree(YOLO_SEG_DIR)

def convert_split_to_yolo_seg(split):
    coco = json.load(open(f'{DATA_DIR}/{split}/_annotations.coco.json'))
    anns_by_image = defaultdict(list)
    for a in coco['annotations']:
        anns_by_image[a['image_id']].append(a)

    img_out = YOLO_SEG_DIR / split / 'images'
    lbl_out = YOLO_SEG_DIR / split / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for im in coco['images']:
        w, h = im['width'], im['height']
        src = Path(DATA_DIR) / split / im['file_name']
        dst = img_out / im['file_name']
        if not dst.exists():
            os.symlink(src, dst)

        lines = []
        for ann in anns_by_image.get(im['id'], []):
            for seg in ann.get('segmentation', []):
                if len(seg) < 6:
                    continue
                coords = []
                for i in range(0, len(seg), 2):
                    x = min(max(seg[i] / w, 0.0), 1.0)
                    y = min(max(seg[i + 1] / h, 0.0), 1.0)
                    coords.append(f'{x:.6f} {y:.6f}')
                lines.append('0 ' + ' '.join(coords))

        (lbl_out / (Path(im['file_name']).stem + '.txt')).write_text('\n'.join(lines))

    return len(coco['images']), sum(len(v) for v in anns_by_image.values())

for split in SPLITS:
    n_img, n_ann = convert_split_to_yolo_seg(split)
    print(f'{split}: {n_img} images, {n_ann} annotations converted')

In [ ]:
data_yaml = f'''
path: {YOLO_SEG_DIR}
train: train/images
val: valid/images
test: test/images
names:
  0: Cracks
'''
yaml_path = YOLO_SEG_DIR / 'data.yaml'
yaml_path.write_text(data_yaml)
print(data_yaml)

## 2. Build the direction-classification dataset

Ultralytics' classification trainer expects `train/<class_name>/*.jpg`,
`val/<class_name>/*.jpg` folders. We build that structure from the
`_direction_labels.csv` files already computed for this dataset (see
`DIRECTION_LABELS.md` for how those labels were derived).

In [ ]:
DIR_DATASET = Path('/content/direction_dataset')
if DIR_DATASET.exists():
    shutil.rmtree(DIR_DATASET)

SPLIT_TO_YOLO_CLS = {'train': 'train', 'valid': 'val', 'test': 'test'}
direction_rows = []

for split in SPLITS:
    with open(f'{DATA_DIR}/{split}/_direction_labels.csv') as f:
        rows = list(csv.DictReader(f))
    out_split = SPLIT_TO_YOLO_CLS[split]
    for r in rows:
        cls_dir = DIR_DATASET / out_split / r['direction']
        cls_dir.mkdir(parents=True, exist_ok=True)
        src = Path(DATA_DIR) / split / r['file_name']
        dst = cls_dir / r['file_name']
        if not dst.exists():
            os.symlink(src, dst)
        r['split'] = split
        direction_rows.append(r)

direction_df = pd.DataFrame(direction_rows)
direction_df['num_annotations'] = direction_df['num_annotations'].astype(int)
direction_df['angle_deg'] = pd.to_numeric(direction_df['angle_deg'], errors='coerce')
direction_df['elongation_ratio'] = pd.to_numeric(direction_df['elongation_ratio'], errors='coerce')
print(direction_df.groupby(['split', 'direction']).size().unstack(fill_value=0))

## 3. Train YOLO-seg — crack detection + shape segmentation

All three models in `SEG_MODELS` (`yolo11n-seg`, `yolo11m-seg`, `yolov8n-seg`) are
trained identically: same data, same image size, same epoch budget, same
hyperparameters. None is a default or a baseline -- comparing across genuinely
different model configurations (two YOLO11 sizes plus a lighter YOLOv8 model) is
exactly what Table 7's pairwise comparisons (Section 10) are designed to test for.

Every training run is followed by a **safety checkpoint**: it bundles whatever's
been trained so far, organized into `segmentation/` and
`direction_classification/` subfolders, into a **uniquely-named** zip and triggers a
real browser download to your computer's normal Downloads folder -- not just a copy
inside this Colab VM. Each checkpoint gets its own filename (e.g.
`crack_model_weights__after_yolo11n_seg.zip`), so your Downloads folder ends up with
one clearly-labeled zip per stage instead of a pile of `(1)`/`(2)`-suffixed
duplicates. If a GitHub token was entered in Section 0, the same weights are also
committed and pushed to `trained_weights/` in this repo at the same point. A
disconnect right after a model finishes never costs you that model's weights, only
the retraining time (skip-if-already-trained will pick it back up if you re-upload
the file into `runs/<run_name>/weights/best.pt`).

In [ ]:
def download_weights_checkpoint(note=''):
    """Bundle every trained model's best.pt found on disk right now into an
    organized, uniquely-named zip, and trigger a real browser download of it to this
    computer's local Downloads folder (colab.files.download() -- not Drive, not just
    this Colab VM's disk). Scans SEG_MODELS/CLS_MODELS dynamically, so this works for
    however many models are configured -- no model is hardcoded as special."""
    ckpt_sources = []
    for seg_model_name in SEG_MODELS:
        size_tag = Path(seg_model_name).stem.split('-')[0]
        ckpt_sources.append(('segmentation', size_tag, f'runs/crack_seg_{size_tag}/weights/best.pt'))
    for cls_model_name in CLS_MODELS:
        size_tag = Path(cls_model_name).stem.split('-')[0]
        ckpt_sources.append(('direction_classification', size_tag, f'runs/crack_direction_cls_{size_tag}/weights/best.pt'))

    ckpt_dir = Path('/content/best_weights_checkpoint')
    if ckpt_dir.exists():
        shutil.rmtree(ckpt_dir)
    ckpt_dir.mkdir(parents=True)

    found = []
    for task, size_tag, src in ckpt_sources:
        src_path = Path(src)
        if not src_path.exists():
            continue
        task_dir = ckpt_dir / task
        task_dir.mkdir(exist_ok=True)
        dst = task_dir / f'{size_tag}_best.pt'
        shutil.copy(src_path, dst)
        found.append(str(dst.relative_to(ckpt_dir)))

    tag = f' ({note})' if note else ''
    print(f'Checkpoint{tag}: {len(found)}/{len(ckpt_sources)} model(s) on disk:')
    for name in found:
        print('  ', name)

    slug = re.sub(r'[^a-z0-9]+', '_', note.lower()).strip('_') if note else 'checkpoint'
    zip_name = f'crack_model_weights__{slug}.zip'
    zip_path = f'/content/{zip_name}'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for f in ckpt_dir.rglob('*.pt'):
            zf.write(f, arcname=str(f.relative_to(ckpt_dir)))

    try:
        from google.colab import files as colab_files
        colab_files.download(zip_path)
        print(f'-> Downloaded to this computer\'s browser Downloads folder as "{zip_name}" '
              f'(organized into segmentation/ and direction_classification/ subfolders inside).')
    except Exception as e:
        print(f'Automatic browser download not available in this environment ({e}). '
              f'The weights are still saved on the Colab VM at {ckpt_dir} and {zip_path}.')
    return ckpt_dir


def push_weights_to_github(note=''):
    """Commit every trained model's best.pt (found on disk right now) into
    trained_weights/ in this repo and push it to BRANCH -- a second, off-runtime
    backup alongside the local zip download above. No-op if
    PUSH_WEIGHTS_TO_GITHUB is False (no token was entered). Never raises: git
    failures are reported and skipped so a backup problem never interrupts training.
    The token is never included in anything printed -- git is driven via explicit
    argv lists (subprocess), and every printed stdout/stderr is scrubbed of it first."""
    if not PUSH_WEIGHTS_TO_GITHUB:
        return
    tag = f' ({note})' if note else ''

    weights_root = Path(DATA_DIR) / 'trained_weights'
    for seg_model_name in SEG_MODELS:
        size_tag = Path(seg_model_name).stem.split('-')[0]
        src = Path('runs') / f'crack_seg_{size_tag}' / 'weights' / 'best.pt'
        if src.exists():
            dst = weights_root / 'segmentation' / f'{size_tag}_best.pt'
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(src, dst)
    for cls_model_name in CLS_MODELS:
        size_tag = Path(cls_model_name).stem.split('-')[0]
        src = Path('runs') / f'crack_direction_cls_{size_tag}' / 'weights' / 'best.pt'
        if src.exists():
            dst = weights_root / 'direction_classification' / f'{size_tag}_best.pt'
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(src, dst)

    def scrub(s):
        return s.replace(GITHUB_TOKEN, '***') if GITHUB_TOKEN else s

    def run_git(args):
        # capture_output + manual returncode check (never raises) so no exception object
        # carrying the token-bearing argv is ever constructed or printed.
        return subprocess.run(['git'] + args, cwd=DATA_DIR, capture_output=True, text=True)

    remote_url = f'https://{GITHUB_TOKEN}@github.com/lestojas/segmentation.git'
    r = run_git(['remote', 'set-url', 'origin', remote_url])
    if r.returncode != 0:
        print(f'GitHub backup{tag}: could not set the authenticated remote '
              f'(token invalid/expired?) -- skipping. {scrub(r.stderr.strip())[:200]}')
        return

    run_git(['config', 'user.email', 'colab-training@users.noreply.github.com'])
    run_git(['config', 'user.name', 'Colab Training Session'])
    run_git(['pull', 'origin', BRANCH])
    run_git(['add', 'trained_weights'])

    status = run_git(['status', '--porcelain'])
    if not status.stdout.strip():
        print(f'GitHub backup{tag}: no weight changes to commit.')
        return

    commit_r = run_git(['commit', '-m', f'Backup trained weights{tag}'])
    if commit_r.returncode != 0:
        print(f'GitHub backup{tag}: nothing committed '
              f'({scrub(commit_r.stdout.strip() or commit_r.stderr.strip())[:200]}).')
        return

    push_r = run_git(['push', 'origin', f'HEAD:{BRANCH}'])
    if push_r.returncode != 0:
        print(f'GitHub backup{tag}: push failed -- {scrub(push_r.stderr.strip())[:300]}')
    else:
        print(f'GitHub backup{tag}: pushed trained_weights/ to {BRANCH}.')

In [ ]:
seg_run_names = {}
for seg_model_name in SEG_MODELS:
    size_tag = Path(seg_model_name).stem.split('-')[0]
    run_name = f'crack_seg_{size_tag}'
    seg_run_names[seg_model_name] = run_name
    weights_path = Path('runs') / run_name / 'weights' / 'best.pt'

    if FORCE_RETRAIN or not weights_path.exists():
        seg_model = YOLO(seg_model_name)
        seg_model.train(
            data=str(yaml_path),
            epochs=SEG_EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH,
            patience=TRAIN_PATIENCE,  # stop early once val performance plateaus -- saves time, doesn't cost accuracy
            cache='ram',       # decode+cache images once instead of re-reading from disk every epoch (pure speed win)
            cos_lr=True,       # cosine LR schedule -- standard fine-tuning setting, usually a small free accuracy gain
            optimizer='auto',  # Ultralytics auto-selects optimizer/lr0/momentum from model size + dataset + epochs
            amp=True,          # mixed precision (default, made explicit) -- ~2x faster on a T4/A100 at equal accuracy
            project='runs',
            name=run_name,
            seed=0,
        )
    else:
        print(f'{weights_path} already exists -- skipping training (FORCE_RETRAIN=False). '
              f'Delete it or set FORCE_RETRAIN=True above to retrain from scratch.')

    download_weights_checkpoint(f'after {size_tag}-seg')
    push_weights_to_github(f'after {size_tag}-seg')

## 4. Train YOLO-cls — crack direction classification

Same idea as Section 3: all three models in `CLS_MODELS` trained identically, each
followed by a safety checkpoint (local zip + optional GitHub push).

In [ ]:
cls_run_names = {}
for cls_model_name in CLS_MODELS:
    size_tag = Path(cls_model_name).stem.split('-')[0]
    run_name = f'crack_direction_cls_{size_tag}'
    cls_run_names[cls_model_name] = run_name
    weights_path = Path('runs') / run_name / 'weights' / 'best.pt'

    if FORCE_RETRAIN or not weights_path.exists():
        cls_model = YOLO(cls_model_name)
        cls_model.train(
            data=str(DIR_DATASET),
            epochs=CLS_EPOCHS,
            imgsz=224,
            batch=BATCH if BATCH != -1 else 64,
            patience=TRAIN_PATIENCE,
            cache='ram',
            cos_lr=True,
            optimizer='auto',
            amp=True,
            project='runs',
            name=run_name,
            seed=0,
        )
    else:
        print(f'{weights_path} already exists -- skipping training (FORCE_RETRAIN=False). '
              f'Delete it or set FORCE_RETRAIN=True above to retrain from scratch.')

    download_weights_checkpoint(f'after {size_tag}-cls')
    push_weights_to_github(f'after {size_tag}-cls')

## 5. Evaluate — crack vs. no-crack detection accuracy

For each seg model: `.val()` gives standard object-detection *and* segmentation
metrics on the held-out **test** split in one call (box precision/recall/mAP,
mask precision/recall/mAP -- kept for Sections 5 and 6, no need to validate twice).
We then build the *image-level* confusion matrix that actually answers "crack vs.
no-crack": for each test image (a near-parity mix of crack and crack-free images --
see the note above), did the model fire at least one crack detection above
`CONF_THRES`? That gives TP/FP/FN/TN and the standard classification metrics
computed from them -- Accuracy = (TP+TN)/(TP+TN+FP+FN), Precision = TP/(TP+FP),
Recall = TP/(TP+FN), Specificity = TN/(TN+FP),
F1 = 2·Precision·Recall/(Precision+Recall).

In [ ]:
test_images_dir = YOLO_SEG_DIR / 'test' / 'images'
test_files = sorted(test_images_dir.glob('*.jpg'))

coco_test = json.load(open(f'{DATA_DIR}/test/_annotations.coco.json'))
gt_has_crack = defaultdict(bool)
for a in coco_test['annotations']:
    gt_has_crack[a['image_id']] = True
filename_to_gt = {im['file_name']: gt_has_crack.get(im['id'], False) for im in coco_test['images']}

seg_results = []
y_true = None
for seg_model_name in SEG_MODELS:
    run_name = seg_run_names[seg_model_name]
    model = YOLO(f'runs/{run_name}/weights/best.pt')
    val = model.val(data=str(yaml_path), split='test', imgsz=IMG_SIZE)

    # Run inference once and keep it -- Section 6 reuses these same Results objects
    # for mask-IoU matching instead of predicting over the test set a second time.
    preds = model.predict(source=[str(p) for p in test_files], conf=CONF_THRES,
                           imgsz=IMG_SIZE, verbose=False)
    y_t, y_p = [], []
    for p in preds:
        fname = Path(p.path).name
        y_t.append(filename_to_gt.get(fname, False))
        y_p.append(len(p.boxes) > 0)
    y_t, y_p = np.array(y_t), np.array(y_p)
    if y_true is None:
        y_true = y_t  # identical for every model -- same test set

    # Explicit confusion matrix: TP = crack present & detected, TN = crack-free &
    # correctly said so, FP = crack-free but a crack was (wrongly) detected,
    # FN = crack present but missed.
    tp = int(np.sum(y_t & y_p)); fn = int(np.sum(y_t & ~y_p))
    fp = int(np.sum(~y_t & y_p)); tn = int(np.sum(~y_t & ~y_p))
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else np.nan

    seg_results.append({
        'model_name': Path(seg_model_name).stem, 'model': model, 'preds': preds, 'y_pred': y_p,
        'tp': tp, 'fn': fn, 'fp': fp, 'tn': tn,
        'accuracy': accuracy, 'precision': precision, 'recall': recall,
        'specificity': specificity, 'f1': f1,
        'box_map50': val.box.map50, 'box_map50_95': val.box.map,
        'box_precision': val.box.mp, 'box_recall': val.box.mr,
        'seg_map50': val.seg.map50, 'seg_map50_95': val.seg.map,
        'seg_precision': val.seg.mp, 'seg_recall': val.seg.mr,
    })

n_pos = int(y_true.sum()); n_neg = int((~y_true).sum())
print(f'Test images: {len(y_true)}  (crack-labeled: {n_pos}, crack-free: {n_neg})')
for r in seg_results:
    print(f"{r['model_name']}: Accuracy={r['accuracy']:.4f}  Precision={r['precision']:.4f}  "
          f"Recall={r['recall']:.4f}  Specificity={r['specificity']:.4f}  F1={r['f1']:.4f}  "
          f"(TP={r['tp']} FN={r['fn']} FP={r['fp']} TN={r['tn']})")

fig, axes = plt.subplots(1, len(seg_results), figsize=(4.5 * len(seg_results), 4))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, seg_results):
    cm = np.array([[r['tp'], r['fn']], [r['fp'], r['tn']]])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred crack', 'Pred no-crack'],
                yticklabels=['Actual crack', 'Actual no-crack'])
    ax.set_title(r['model_name'])
plt.suptitle('Crack vs. no-crack — confusion matrix per model'); plt.tight_layout(); plt.show()

## 6. Evaluate — crack shape segmentation accuracy

Mask mAP/precision/recall already came out of Section 5's `val()` call for each
model. Here we additionally compute the mean IoU between each predicted mask and
its best-matching ground-truth mask (reusing Section 5's predictions -- no second
inference pass), which is a more intuitive "how good is the crack shape" number.

In [ ]:
def polygon_mask(segmentation, w, h):
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)
    for seg in segmentation:
        pts = list(zip(seg[0::2], seg[1::2]))
        if len(pts) >= 3:
            draw.polygon(pts, fill=1)
    return np.array(mask, dtype=bool)

def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / union if union > 0 else 0.0

id_by_name = {im['file_name']: im for im in coco_test['images']}
anns_by_image_test = defaultdict(list)
for a in coco_test['annotations']:
    anns_by_image_test[a['image_id']].append(a)

def best_matched_ious(preds):
    """Given a list of Results already predicted over test_files, return, for every
    ground-truth crack instance (in a fixed deterministic order), the best-matching
    IoU. Calling this with different models' predictions yields arrays that line up
    index-for-index on the same ground-truth instances -- exactly what Table 7's
    paired tests need."""
    ious_list = []
    for p in preds:
        fname = Path(p.path).name
        im_meta = id_by_name.get(fname)
        if im_meta is None:
            continue
        w, h = im_meta['width'], im_meta['height']
        gt_masks = [polygon_mask(a['segmentation'], w, h) for a in anns_by_image_test.get(im_meta['id'], [])]
        if not gt_masks:
            continue
        if p.masks is None:
            ious_list.extend([0.0] * len(gt_masks))  # missed every ground-truth crack in this image
            continue
        # p.masks.xy gives polygon vertices already in original-image pixel coordinates,
        # which avoids any ambiguity about the resolution of p.masks.data.
        pred_masks = [polygon_mask([poly.reshape(-1).tolist()], w, h) for poly in p.masks.xy]
        used = set()
        for gt in gt_masks:
            best_iou, best_j = 0.0, None
            for j, pm in enumerate(pred_masks):
                if j in used:
                    continue
                iou = mask_iou(gt, pm)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_j is not None:
                used.add(best_j)
            ious_list.append(best_iou)
    return np.array(ious_list)

for r in seg_results:
    r['ious'] = best_matched_ious(r['preds'])
    print(f"{r['model_name']}: Mean mask IoU={r['ious'].mean():.4f}  Median={np.median(r['ious']):.4f}  "
          f"mAP50={r['seg_map50']:.4f}  Precision={r['seg_precision']:.4f}  Recall={r['seg_recall']:.4f}")

fig, axes = plt.subplots(1, len(seg_results), figsize=(5 * len(seg_results), 4))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, seg_results):
    ax.hist(r['ious'], bins=20, edgecolor='black')
    ax.set_title(f"{r['model_name']} — mean IoU {r['ious'].mean():.3f}")
    ax.set_xlabel('Mask IoU'); ax.set_ylabel('Number of crack instances')
plt.suptitle('Segmentation shape accuracy — IoU distribution per model'); plt.tight_layout(); plt.show()

## 7. Evaluate — crack direction classification accuracy

In [ ]:
test_dir_dataset = DIR_DATASET / 'test'
y_true_dir, files = [], []
for cls_name in sorted(os.listdir(test_dir_dataset)):
    for f in (test_dir_dataset / cls_name).glob('*.jpg'):
        files.append(f)
        y_true_dir.append(cls_name)
y_true_dir_arr = np.array(y_true_dir)

cls_results = []
for cls_model_name in CLS_MODELS:
    run_name = cls_run_names[cls_model_name]
    model = YOLO(f'runs/{run_name}/weights/best.pt')
    val = model.val(data=str(DIR_DATASET), split='test')

    class_names = model.names
    preds = model.predict(source=[str(f) for f in files], imgsz=224, verbose=False)
    y_pred_dir_arr = np.array([class_names[int(p.probs.top1)] for p in preds])

    accuracy = (y_true_dir_arr == y_pred_dir_arr).mean()
    report = classification_report(y_true_dir, list(y_pred_dir_arr), output_dict=True, digits=4)
    macro = report['macro avg']

    cls_results.append({
        'model_name': Path(cls_model_name).stem, 'model': model, 'y_pred_dir': y_pred_dir_arr,
        'accuracy': accuracy, 'top1': val.top1, 'top5': val.top5,
        'macro_precision': macro['precision'], 'macro_recall': macro['recall'],
        'macro_f1': macro['f1-score'],
    })
    print(f"{Path(cls_model_name).stem}: Top-1 accuracy={accuracy:.4f}  "
          f"Macro precision={macro['precision']:.4f}  Macro recall={macro['recall']:.4f}  "
          f"Macro F1={macro['f1-score']:.4f}")

labels_order = sorted(set(y_true_dir) | {d for r in cls_results for d in r['y_pred_dir']})
fig, axes = plt.subplots(1, len(cls_results), figsize=(5.5 * len(cls_results), 4.5))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, cls_results):
    cm = confusion_matrix(y_true_dir, r['y_pred_dir'], labels=labels_order)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=labels_order, yticklabels=labels_order)
    ax.set_title(r['model_name']); ax.set_xlabel('Predicted direction'); ax.set_ylabel('True direction')
plt.suptitle('Direction classification — confusion matrix per model'); plt.tight_layout(); plt.show()

## 8. Combined summary

A quick recap across every model -- the full breakdown lives in Tables 3, 5, and 6
(Section 10).

In [ ]:
summary_rows = []
for r in seg_results:
    summary_rows.append({'Model': r['model_name'], 'Task': 'Crack vs. no-crack detection',
                          'Metric': 'Image-level accuracy', 'Value': f"{r['accuracy']:.4f}"})
    summary_rows.append({'Model': r['model_name'], 'Task': 'Crack vs. no-crack detection',
                          'Metric': 'Box mAP50', 'Value': f"{r['box_map50']:.4f}"})
    summary_rows.append({'Model': r['model_name'], 'Task': 'Crack shape segmentation',
                          'Metric': 'Mean mask IoU', 'Value': f"{r['ious'].mean():.4f}"})
    summary_rows.append({'Model': r['model_name'], 'Task': 'Crack shape segmentation',
                          'Metric': 'Mask mAP50', 'Value': f"{r['seg_map50']:.4f}"})
for r in cls_results:
    summary_rows.append({'Model': r['model_name'], 'Task': 'Crack direction classification',
                          'Metric': 'Top-1 accuracy', 'Value': f"{r['accuracy']:.4f}"})
summary = pd.DataFrame(summary_rows)
summary

## 9. Visualize sample predictions

A handful of test images with the predicted crack mask overlaid and the predicted
direction label as the title, one column per model pair (matching seg/cls models by
index in `SEG_MODELS`/`CLS_MODELS`), for a quick sanity check.

In [ ]:
sample_files = random.sample(test_files, min(3, len(test_files)))
model_pairs = list(zip(seg_results, cls_results))

fig, axes = plt.subplots(len(sample_files), len(model_pairs),
                          figsize=(5 * len(model_pairs), 5 * len(sample_files)), squeeze=False)
for row, fpath in enumerate(sample_files):
    for col, (seg_r, cls_r) in enumerate(model_pairs):
        seg_pred = seg_r['model'].predict(source=str(fpath), conf=CONF_THRES, imgsz=IMG_SIZE, verbose=False)[0]
        dir_pred = cls_r['model'].predict(source=str(fpath), imgsz=224, verbose=False)[0]
        annotated = seg_pred.plot()[:, :, ::-1]  # BGR -> RGB
        ax = axes[row][col]
        ax.imshow(annotated)
        dir_name = dir_pred.names[int(dir_pred.probs.top1)]
        ax.set_title(f"{seg_r['model_name']}\n{fpath.name}\npredicted direction: {dir_name} "
                     f"({float(dir_pred.probs.top1conf):.2f})", fontsize=8)
        ax.axis('off')
plt.tight_layout(); plt.show()

## 10. Export research-paper-ready tables (descriptive & inferential statistics)

Each table is built as a `pandas` DataFrame, displayed inline, and then written to
`/content/paper_tables/` as both `.csv` (spreadsheet-ready) and `.tex` (a LaTeX
`tabular` you can `\input{}` straight into a paper), bundled into `paper_tables.zip`
for download.

- **Table 0 (descriptive, Methods)** — image counts per split (total, cracked,
  crack-free): the high-level dataset-size table a Methods/Data section needs.
- **Table 1 (descriptive, Results)** — the held-out **test** split's composition in
  detail: crack-instance counts and crack size by direction, plus a `No Crack` row.
  Matches the images the model is actually scored on below.
- **Table 2 (descriptive)** — crack detection confusion matrix (TP/FN/FP/TN), one row
  per model.
- **Table 3 (descriptive)** — crack detection *image-level* metrics (Accuracy,
  Precision, Recall, Specificity, F1), computed from Table 2.
- **Table 4 (descriptive)** — crack detection *box-level* metrics (Ultralytics' own
  box-IoU-matched precision/recall/mAP) -- a different question from Table 3, kept
  separate rather than mixed into it.
- **Table 5 (descriptive)** — crack segmentation performance (mean mask IoU, mask
  precision/recall/mAP).
- **Table 6 (descriptive)** — direction classification performance (top-1 accuracy,
  macro precision/recall/F1).
- Tables 2-6 each have one column per model actually configured in
  `SEG_MODELS`/`CLS_MODELS`, named for the model itself -- no "main"/"baseline"
  distinction anywhere.
- **Table 7 (inferential)** — a **symmetric** paired significance test between
  *every pair* of models, for each task (McNemar's for detection/direction,
  Wilcoxon signed-rank for segmentation IoU, both two-sided). With three models
  that's 3 pairwise comparisons x 3 tasks = 9 rows. No model is treated as a
  reference; a `Higher-performing model` column reports which side scored higher
  on the point estimate, purely descriptively, computed *after* the test.

In [ ]:
PAPER_TABLES_DIR = Path('/content/paper_tables')
PAPER_TABLES_DIR.mkdir(exist_ok=True)
exported_tables = {}  # name -> DataFrame; exported to CSV/LaTeX in the final cell of this section

def register_table(name, df):
    exported_tables[name] = df
    return df

### Table 0 — Dataset split summary (for the Methods section)

How many images are in each split, and how many are crack vs. crack-free -- the level
of detail a Methods/Data section typically needs, without the per-direction breakdown.

In [ ]:
all_coco = {split: json.load(open(f'{DATA_DIR}/{split}/_annotations.coco.json')) for split in SPLITS}

area_records = []
no_crack_counts = {split: 0 for split in SPLITS}
for split, coco in all_coco.items():
    imgs_by_id = {im['id']: im for im in coco['images']}
    for im in coco['images']:
        if im.get('direction') == 'None':
            no_crack_counts[split] += 1
    for a in coco['annotations']:
        im = imgs_by_id[a['image_id']]
        area_records.append({
            'split': split,
            'direction': im.get('direction', 'Unknown'),
            'area_px': a['area'],
            'area_pct': 100 * a['area'] / (im['width'] * im['height']),
        })
area_df = pd.DataFrame(area_records)

table0 = pd.DataFrame([
    {'split': split, 'n_images': len(all_coco[split]['images']),
     'n_cracked_images': len(all_coco[split]['images']) - no_crack_counts[split],
     'n_no_crack_images': no_crack_counts[split]}
    for split in SPLITS
])
totals = table0[['n_images', 'n_cracked_images', 'n_no_crack_images']].sum().to_dict()
table0 = pd.concat([table0, pd.DataFrame([{'split': 'All', **totals}])], ignore_index=True)
register_table('Table0_dataset_split_summary', table0)
display(table0)

### Table 1 — Test split composition (for the Results section)

The fine-grained breakdown of the held-out **test** split only -- crack-instance
counts and crack size (as % of image area) by direction, plus a `No Crack` row for
the negatives, matching the actual images the model is evaluated on in the results
below. (See Table 0 above for train/valid/test totals.)

In [ ]:
image_counts = direction_df.groupby(['split', 'direction']).size().rename('n_images').reset_index()
ann_counts = (direction_df.groupby(['split', 'direction'])['num_annotations'].sum()
              .rename('n_annotations').reset_index())
area_stats = (area_df.groupby(['split', 'direction'])['area_pct']
              .agg(['count', 'mean', 'std', 'median', 'min', 'max']).reset_index())
area_stats.columns = ['split', 'direction', 'n_instances', 'mean_area_pct', 'sd_area_pct',
                       'median_area_pct', 'min_area_pct', 'max_area_pct']

table1_cracked = (image_counts.merge(ann_counts, on=['split', 'direction'])
                  .merge(area_stats, on=['split', 'direction']))

no_crack_rows = pd.DataFrame([
    {'split': split, 'direction': 'No Crack', 'n_images': no_crack_counts[split], 'n_annotations': 0,
     'n_instances': 0, 'mean_area_pct': np.nan, 'sd_area_pct': np.nan, 'median_area_pct': np.nan,
     'min_area_pct': np.nan, 'max_area_pct': np.nan}
    for split in SPLITS
])

table1_all_splits = (pd.concat([table1_cracked, no_crack_rows], ignore_index=True)
                      .sort_values(['split', 'direction']).reset_index(drop=True).round(3))
table1 = (table1_all_splits[table1_all_splits['split'] == 'test']
          .drop(columns='split').reset_index(drop=True))
register_table('Table1_test_split_composition', table1)
display(table1)

### Table 2 — Crack detection confusion matrix

The raw TP/FN/FP/TN counts behind every image-level detection metric below, one row
per model, over the full test set (crack + crack-free images).

In [ ]:
cm_rows = [{'Model': r['model_name'], 'TP': r['tp'], 'FN': r['fn'], 'FP': r['fp'], 'TN': r['tn'],
            'N': len(y_true)} for r in seg_results]
table2_cm = pd.DataFrame(cm_rows)
register_table('Table2_detection_confusion_matrix', table2_cm)
display(table2_cm)

### Table 3 — Crack detection: image-level metrics

Accuracy/Precision/Recall/Specificity/F1, computed from Table 2's confusion matrix --
answers "crack present or not, per image." One column per model, named for the
model itself.

In [ ]:
det_metrics = [('Accuracy', 'accuracy'), ('Precision', 'precision'), ('Recall', 'recall'),
                ('Specificity', 'specificity'), ('F1-score', 'f1')]
table3_det_img = pd.DataFrame([
    {'Metric': label, **{r['model_name']: r[key] for r in seg_results}}
    for label, key in det_metrics
]).round(4)
register_table('Table3_detection_image_level_metrics', table3_det_img)
display(table3_det_img)

### Table 4 — Crack detection: box-level metrics

Ultralytics' own object-detection metrics -- whether individual predicted boxes line
up with ground-truth boxes by IoU. A different question from Table 3 (box-level vs.
image-level), so read separately rather than side by side.

In [ ]:
box_metrics = [('Box precision', 'box_precision'), ('Box recall', 'box_recall'),
                ('Box mAP50', 'box_map50'), ('Box mAP50-95', 'box_map50_95')]
table4_det_box = pd.DataFrame([
    {'Metric': label, **{r['model_name']: r[key] for r in seg_results}}
    for label, key in box_metrics
]).round(4)
register_table('Table4_detection_box_level_metrics', table4_det_box)
display(table4_det_box)

### Table 5 — Crack segmentation performance (descriptive)

In [ ]:
table5_seg = pd.DataFrame([
    {'Metric': 'Mean mask IoU', **{r['model_name']: r['ious'].mean() for r in seg_results}},
    {'Metric': 'Mask precision', **{r['model_name']: r['seg_precision'] for r in seg_results}},
    {'Metric': 'Mask recall', **{r['model_name']: r['seg_recall'] for r in seg_results}},
    {'Metric': 'Mask mAP50', **{r['model_name']: r['seg_map50'] for r in seg_results}},
    {'Metric': 'Mask mAP50-95', **{r['model_name']: r['seg_map50_95'] for r in seg_results}},
]).round(4)
register_table('Table5_segmentation_performance', table5_seg)
display(table5_seg)

### Table 6 — Direction classification performance (descriptive)

In [ ]:
table6_dir = pd.DataFrame([
    {'Metric': 'Top-1 accuracy', **{r['model_name']: r['accuracy'] for r in cls_results}},
    {'Metric': 'Macro precision', **{r['model_name']: r['macro_precision'] for r in cls_results}},
    {'Metric': 'Macro recall', **{r['model_name']: r['macro_recall'] for r in cls_results}},
    {'Metric': 'Macro F1', **{r['model_name']: r['macro_f1'] for r in cls_results}},
]).round(4)
register_table('Table6_direction_performance', table6_dir)
display(table6_dir)

### Table 7 — Are the models significantly different from each other? (inferential)

Every model was trained and scored on the *exact same data and test set*, so every
comparison here is **paired** -- both models are scored on the identical images /
crack instances, which controls for test-set variability. This is a genuinely
**symmetric, two-sided test for every pair of models**: it asks "are these two
significantly different", not "is A better than B" -- no model is singled out as a
reference.

| Task | Test | Why this test |
|---|---|---|
| Detection | McNemar's test (two-sided) | standard test for two classifiers scored on the same items |
| Segmentation | Wilcoxon signed-rank (two-sided) | paired, non-parametric, robust to IoU being bounded/skewed |
| Direction | McNemar's test (two-sided) | same as detection -- both are per-image correct/incorrect outcomes |

**How to read it:** `p < 0.05` means the two models' paired outcomes differ
significantly -- a real difference, not just noise from a particular test split. The
`Higher-performing model` column is purely descriptive (computed from the two point
estimates *after* the test runs), not part of the hypothesis being tested. With
three models compared, every task gets 3 pairwise rows -- 9 rows total.

In [ ]:
def mcnemar_test(correct_a, correct_b):
    """Two-sided McNemar's test: are two models' per-item correctness significantly
    different on the same paired items? Uses the exact binomial form when the number of
    discordant pairs is small (n<25, the standard recommendation), and the
    continuity-corrected chi-square approximation otherwise. Symmetric in A/B -- neither
    side is treated as a reference/baseline."""
    correct_a, correct_b = np.asarray(correct_a), np.asarray(correct_b)
    b = int(np.sum(correct_a & ~correct_b))   # A right, B wrong
    c = int(np.sum(~correct_a & correct_b))   # A wrong, B right
    n = b + c
    if n == 0:
        return np.nan, 1.0, b, c
    if n < 25:
        p = stats.binomtest(b, n, 0.5, alternative='two-sided').pvalue
        stat = np.nan
    else:
        stat = (abs(b - c) - 1) ** 2 / n
        p = 1 - stats.chi2.cdf(stat, df=1)
    return stat, p, b, c

def higher_performing(label_a, score_a, label_b, score_b):
    if score_a == score_b:
        return 'Tie'
    return label_a if score_a > score_b else label_b

rows = []

for ra, rb in itertools.combinations(seg_results, 2):
    correct_a, correct_b = (y_true == ra['y_pred']), (y_true == rb['y_pred'])
    stat, p, b, c = mcnemar_test(correct_a, correct_b)
    rows.append({'Task': 'Crack detection', 'Model A': ra['model_name'], 'Model B': rb['model_name'],
                 'Model A score': round(ra['accuracy'], 4), 'Model B score': round(rb['accuracy'], 4),
                 'Test': "McNemar's test", 'Statistic': stat, 'p_value': p,
                 'Significant (p<0.05)': p < 0.05,
                 'Higher-performing model': higher_performing(ra['model_name'], ra['accuracy'],
                                                                rb['model_name'], rb['accuracy'])})

for ra, rb in itertools.combinations(seg_results, 2):
    iou_diffs = ra['ious'] - rb['ious']
    if np.any(iou_diffs != 0):
        seg_stat, seg_p = stats.wilcoxon(iou_diffs, alternative='two-sided')
    else:
        seg_stat, seg_p = np.nan, np.nan
    rows.append({'Task': 'Crack segmentation', 'Model A': ra['model_name'], 'Model B': rb['model_name'],
                 'Model A score': round(ra['ious'].mean(), 4), 'Model B score': round(rb['ious'].mean(), 4),
                 'Test': 'Wilcoxon signed-rank', 'Statistic': seg_stat, 'p_value': seg_p,
                 'Significant (p<0.05)': seg_p < 0.05,
                 'Higher-performing model': higher_performing(ra['model_name'], ra['ious'].mean(),
                                                                rb['model_name'], rb['ious'].mean())})

for ra, rb in itertools.combinations(cls_results, 2):
    correct_a = y_true_dir_arr == ra['y_pred_dir']
    correct_b = y_true_dir_arr == rb['y_pred_dir']
    stat, p, b, c = mcnemar_test(correct_a, correct_b)
    rows.append({'Task': 'Direction classification', 'Model A': ra['model_name'], 'Model B': rb['model_name'],
                 'Model A score': round(ra['accuracy'], 4), 'Model B score': round(rb['accuracy'], 4),
                 'Test': "McNemar's test", 'Statistic': stat, 'p_value': p,
                 'Significant (p<0.05)': p < 0.05,
                 'Higher-performing model': higher_performing(ra['model_name'], ra['accuracy'],
                                                                rb['model_name'], rb['accuracy'])})

table7 = pd.DataFrame(rows)
table7[['Statistic', 'p_value']] = table7[['Statistic', 'p_value']].round(4)
register_table('Table7_pairwise_model_comparisons', table7)
display(table7)

### Export tables to CSV + LaTeX and download

In [ ]:
for name, df in exported_tables.items():
    df.to_csv(PAPER_TABLES_DIR / f'{name}.csv', index=False)
    try:
        (PAPER_TABLES_DIR / f'{name}.tex').write_text(
            df.to_latex(index=False, float_format='%.4f', na_rep='--'))
    except Exception as e:
        print(f'Could not export {name} to LaTeX ({e}); CSV was still written.')

print(f'Exported {len(exported_tables)} tables to {PAPER_TABLES_DIR}:')
for f in sorted(PAPER_TABLES_DIR.iterdir()):
    print(' ', f.name)

zip_path = '/content/paper_tables.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in PAPER_TABLES_DIR.iterdir():
        zf.write(f, arcname=f.name)

try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
    print('-> Downloaded to this computer\'s browser Downloads folder as "paper_tables.zip" '
          '(one .csv + .tex per table inside).')
except Exception as e:
    print(f'Automatic browser download not available in this environment ({e}). '
          f'The files are still saved on the Colab VM at {PAPER_TABLES_DIR} and {zip_path}.')

## 11. Download trained weights

Bundles the best checkpoint (`best.pt`) for every model into one zip and triggers a
browser download, the same way Section 10 does for the paper tables. Also pushes a
final copy to `trained_weights/` in this GitHub repo if a token was entered in
Section 0. Reuses the same `download_weights_checkpoint()` / `push_weights_to_github()`
the training cells in Sections 3-4 already called after each model, so this is just
a final, complete copy of both.

In [ ]:
weights_checkpoint_dir = download_weights_checkpoint('final -- all sections done')
push_weights_to_github('final -- all sections done')

## 12. (Optional) Also save weights & tables to Google Drive

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    out_dir = '/content/drive/MyDrive/crack_models'
    weights_out_dir = f'{out_dir}/weights'
    if os.path.isdir(weights_out_dir):
        shutil.rmtree(weights_out_dir)
    shutil.copytree(weights_checkpoint_dir, weights_out_dir)
    tables_out_dir = f'{out_dir}/paper_tables'
    if os.path.isdir(tables_out_dir):
        shutil.rmtree(tables_out_dir)
    shutil.copytree(PAPER_TABLES_DIR, tables_out_dir)
    print(f'Saved weights and paper tables to {out_dir}')
else:
    print('SAVE_TO_DRIVE is False — skipping. Weights remain at runs/<run_name>/weights/best.pt '
          f'for each trained model, and exported tables remain at {PAPER_TABLES_DIR} / '
          '/content/paper_tables.zip for this session (Section 11 above already bundled '
          'and downloaded the weights to your browser).')